# RMS Zernike SPGD 优化日志分析

用于分析 data/rms_zernike 目录下每次 SPGD 运行的优化日志，可视化 RMS 历史、Zernike 系数演化、学习率调度等。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import ast
import re

In [ ]:
# 查找所有运行目录
data_root = Path("../data/rms_zernike")
run_dirs = sorted([d for d in data_root.iterdir() if d.is_dir()], key=lambda x: x.stat().st_mtime, reverse=True)
run_names = [d.name for d in run_dirs]
print(f"找到 {len(run_dirs)} 个运行目录:")
for name in run_names[:5]:
    print(f"  - {name}")
if len(run_names) > 5:
    print(f"  ... 共 {len(run_names)} 个")

In [ ]:
def parse_array(s):
    """解析CSV中的数组字符串"""
    if pd.isna(s) or s == '':
        return None
    try:
        # 尝试直接解析
        return ast.literal_eval(s)
    except:
        try:
            # 尝试解析numpy数组字符串
            s_clean = s.replace('[', '').replace(']', '').replace('\n', ' ')
            return np.fromstring(s_clean, sep=' ')
        except:
            return None

def load_run_csv(run_dir: Path):
    """加载运行目录中的CSV文件"""
    csv_files = list(run_dir.glob("*.csv"))
    if not csv_files:
        return None
    
    csv_file = csv_files[0]  # 取第一个CSV文件
    print(f"加载: {csv_file.name}")
    
    # 读取CSV，保留原始字符串以便后续解析
    df = pd.read_csv(csv_file, dtype=str)
    
    # 解析关键列
    def parse_rms(x):
        try: return float(x)
        except: return np.nan
    
    df['rms_num'] = df['rms'].apply(parse_rms)
    df['epoch_num'] = df['_epoch'].apply(lambda x: int(x) if x else 0)
    df['delta_num'] = df['delta'].apply(lambda x: float(x) if x else np.nan)
    df['gamma_num'] = df['_gamma'].apply(lambda x: float(x) if x else np.nan)
    df['diff_num'] = df['_diff'].apply(lambda x: float(x) if x else np.nan)
    
    return df, csv_file.name

In [ ]:
def parse_zernike_coeffs(coef_str):
    """解析Zernike系数字符串"""
    if pd.isna(coef_str) or coef_str == '':
        return None
    try:
        arr = ast.literal_eval(coef_str)
        return np.array(arr) if arr else None
    except:
        return None

def extract_zernike_matrix(df):
    """提取所有epoch的Zernike系数矩阵"""
    coef_list = []
    for _, row in df.iterrows():
        coef = parse_zernike_coeffs(row['_c'])
        if coef is not None:
            coef_list.append(coef)
    if coef_list:
        return np.array(coef_list)
    return None

def extract_pos_neg_coeffs(df):
    """提取正负扰动系数"""
    pos_list, neg_list = [], []
    for _, row in df.iterrows():
        if '_pos_c' in row and pd.notna(row['_pos_c']):
            coef = parse_zernike_coeffs(row['_pos_c'])
            if coef is not None:
                pos_list.append(coef)
        if '_neg_c' in row and pd.notna(row['_neg_c']):
            coef = parse_zernike_coeffs(row['_neg_c'])
            if coef is not None:
                neg_list.append(coef)
    return np.array(pos_list) if pos_list else None, np.array(neg_list) if neg_list else None

In [ ]:
# 运行选择器
run_dropdown = widgets.Dropdown(
    options=run_names,
    value=run_names[0] if run_names else None,
    description='运行:',
    style={'description_width': '60px'}
)

load_btn = widgets.Button(description='加载数据', button_style='primary')
output_load = widgets.Output()

current_df = {'df': None, 'run_name': None}

def on_load_click(b):
    with output_load:
        clear_output()
        if run_dropdown.value:
            run_dir = data_root / run_dropdown.value
            result = load_run_csv(run_dir)
            if result:
                df, name = result
                current_df['df'] = df
                current_df['run_name'] = name
                print(f"✓ 加载完成: {len(df)} epochs")
                print(f"  RMS范围: {df['rms_num'].min():.4f} - {df['rms_num'].max():.4f}")
                print(f"  最佳RMS: {df['rms_num'].min():.4f} @ epoch {df.loc[df['rms_num'].idxmin(), 'epoch_num']}")
            else:
                print("✗ 未找到CSV文件")

load_btn.on_click(on_load_click)

display(widgets.HBox([run_dropdown, load_btn]))
display(output_load)

In [ ]:
# 可视化: RMS历史 + 学习率调度
def plot_rms_history(df):
    """绘制RMS优化历史"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # RMS曲线
    ax = axes[0, 0]
    epochs = df['epoch_num'].values
    rms_values = df['rms_num'].values
    ax.plot(epochs, rms_values, 'b-', alpha=0.7, linewidth=0.5)
    ax.scatter(epochs, rms_values, c='blue', s=2, alpha=0.3)
    
    # 标记最佳点
    best_idx = df['rms_num'].idxmin()
    best_epoch = df.loc[best_idx, 'epoch_num']
    best_rms = df.loc[best_idx, 'rms_num']
    ax.axhline(y=best_rms, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_rms:.4f}')
    ax.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.5)
    ax.scatter([best_epoch], [best_rms], c='red', s=100, zorder=5, marker='*')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('RMS')
    ax.set_title('RMS 优化历史')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 学习率演化
    ax = axes[0, 1]
    ax.plot(epochs, df['gamma_num'].values, 'g-', linewidth=1)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate (γ)')
    ax.set_title('学习率调度')
    ax.grid(True, alpha=0.3)
    
    # Delta演化
    ax = axes[1, 0]
    ax.plot(epochs, df['delta_num'].values, 'm-', linewidth=1)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Delta (扰动幅度)')
    ax.set_title('扰动幅度调度')
    ax.grid(True, alpha=0.3)
    
    # 梯度差异
    ax = axes[1, 1]
    ax.plot(epochs, df['diff_num'].values, 'orange', linewidth=0.5, alpha=0.7)
    ax.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Gradient (pos - neg)')
    ax.set_title('SPGD 梯度估计')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 创建按钮
plot_rms_btn = widgets.Button(description='绘制RMS历史', button_style='success')
output_plot = widgets.Output()

def on_plot_click(b):
    with output_plot:
        clear_output()
        if current_df['df'] is not None:
            plot_rms_history(current_df['df'])
        else:
            print("请先加载数据")

plot_rms_btn.on_click(on_plot_click)
display(plot_rms_btn)
display(output_plot)

In [ ]:
# 可视化: Zernike系数演化
def plot_zernike_evolution(df):
    """绘制Zernike系数演化"""
    coef_matrix = extract_zernike_matrix(df)
    if coef_matrix is None:
        print("无法解析Zernike系数")
        return
    
    n_modes = coef_matrix.shape[1]
    n_cols = min(4, n_modes)
    n_rows = (n_modes + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    axes = axes.flatten()
    
    epochs = df['epoch_num'].values
    
    for i in range(n_modes):
        ax = axes[i]
        ax.plot(epochs, coef_matrix[:, i], linewidth=0.8)
        ax.set_title(f'Zernike #{i+1}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Coefficient')
        ax.grid(True, alpha=0.3)
    
    # 隐藏多余的子图
    for i in range(n_modes, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # 打印最终系数
    print("\n最终Zernike系数:")
    final_coefs = coef_matrix[-1] if len(coef_matrix) > 0 else None
    if final_coefs is not None:
        for i, c in enumerate(final_coefs):
            print(f"  Z{i+1}: {c:.6f}")

In [ ]:
# 创建按钮
plot_zern_btn = widgets.Button(description='绘制Zernike演化', button_style='info')
output_zern = widgets.Output()

def on_zern_click(b):
    with output_zern:
        clear_output()
        if current_df['df'] is not None:
            plot_zernike_evolution(current_df['df'])
        else:
            print("请先加载数据")

plot_zern_btn.on_click(on_zern_click)
display(plot_zern_btn)
display(output_zern)

In [ ]:
# 统计信息
def show_stats(df):
    """显示统计信息"""
    print("=" * 50)
    print("SPGD 优化统计信息")
    print("=" * 50)
    
    print(f"总迭代次数: {len(df)}")
    print(f"初始RMS: {df['rms_num'].iloc[0]:.4f}")
    print(f"最终RMS: {df['rms_num'].iloc[-1]:.4f}")
    
    best_idx = df['rms_num'].idxmin()
    best_epoch = df.loc[best_idx, 'epoch_num']
    best_rms = df.loc[best_idx, 'rms_num']
    print(f"最佳RMS: {best_rms:.4f} @ epoch {best_epoch}")
    
    improvement = (df['rms_num'].iloc[0] - best_rms) / df['rms_num'].iloc[0] * 100
    print(f"RMS改善: {improvement:.1f}%")
    
    # 学习率统计
    print(f"\n学习率范围: {df['gamma_num'].min():.6f} - {df['gamma_num'].max():.6f}")
    print(f"Delta范围: {df['delta_num'].min():.6f} - {df['delta_num'].max():.6f}")
    
    # 收敛分析
    rms_values = df['rms_num'].values
    # 计算滑动平均
    window = min(50, len(rms_values))
    if window > 10:
        ma = np.convolve(rms_values, np.ones(window)/window, mode='valid')
        final_ma = np.mean(ma[-10:])
        print(f"\n最后{window} epoch滑动平均RMS: {final_ma:.4f}")
    
    # 早停检测
    early_stop_epochs = df[df['rms_num'] < 0.12]['epoch_num']
    if len(early_stop_epochs) > 0:
        print(f"达到早停阈值(0.12)的epoch: {early_stop_epochs.iloc[0]}")
    else:
        print("未达到早停阈值(0.12)")
    
    print("=" * 50)

In [ ]:
# 创建按钮
stats_btn = widgets.Button(description='显示统计', button_style='warning')
output_stats = widgets.Output()

def on_stats_click(b):
    with output_stats:
        clear_output()
        if current_df['df'] is not None:
            show_stats(current_df['df'])
        else:
            print("请先加载数据")

stats_btn.on_click(on_stats_click)
display(stats_btn)
display(output_stats)

In [ ]:
# 多运行对比
def compare_runs(run_indices):
    """对比多个运行的RMS历史"""
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(run_indices)))
    
    for idx, i in enumerate(run_indices):
        if i >= len(run_dirs):
            continue
        run_dir = run_dirs[i]
        result = load_run_csv(run_dir)
        if result:
            df, name = result
            ax.plot(df['epoch_num'].values, df['rms_num'].values, 
                   color=colors[idx], alpha=0.7, linewidth=1, label=name[:8])
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('RMS')
    ax.set_title('多运行RMS对比')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# 运行对比选择器
compare_btn = widgets.Button(description='对比最近3个运行', button_style='primary')
output_compare = widgets.Output()

def on_compare_click(b):
    with output_compare:
        clear_output()
        compare_runs([0, 1, 2])

compare_btn.on_click(on_compare_click)
display(compare_btn)
display(output_compare)

In [ ]:
# 动画与GIF导出
import matplotlib.animation as animation
from PIL import Image
import io

def create_rms_animation(df, save_path=None, fps=10, skip_frames=1):
    """创建RMS历史演化动画"""
    epochs = df['epoch_num'].values
    rms_values = df['rms_num'].values
    gamma_values = df['gamma_num'].values
    delta_values = df['delta_num'].values
    
    # 降采样以加速动画
    indices = list(range(0, len(epochs), skip_frames))
    if indices[-1] != len(epochs) - 1:
        indices.append(len(epochs) - 1)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # 初始化
    line_rms, = axes[0, 0].plot([], [], 'b-', linewidth=1)
    scatter_rms = axes[0, 0].scatter([], [], c='blue', s=10, alpha=0.5)
    best_marker = axes[0, 0].scatter([], [], c='red', s=100, marker='*', zorder=5)
    axes[0, 0].set_xlim(0, max(epochs))
    axes[0, 0].set_ylim(min(rms_values)*0.9, max(rms_values)*1.1)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('RMS')
    axes[0, 0].set_title('RMS 优化历史')
    axes[0, 0].grid(True, alpha=0.3)
    
    line_gamma, = axes[0, 1].plot([], [], 'g-', linewidth=1)
    axes[0, 1].set_xlim(0, max(epochs))
    axes[0, 1].set_ylim(0, max(gamma_values)*1.1)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Learning Rate')
    axes[0, 1].set_title('学习率调度')
    axes[0, 1].grid(True, alpha=0.3)
    
    line_delta, = axes[1, 0].plot([], [], 'm-', linewidth=1)
    axes[1, 0].set_xlim(0, max(epochs))
    axes[1, 0].set_ylim(0, max(delta_values)*1.1)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Delta')
    axes[1, 0].set_title('扰动幅度调度')
    axes[1, 0].grid(True, alpha=0.3)
    
    line_diff, = axes[1, 1].plot([], [], 'orange', linewidth=0.5, alpha=0.7)
    axes[1, 1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[1, 1].set_xlim(0, max(epochs))
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Gradient')
    axes[1, 1].set_title('SPGD 梯度估计')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # 找到最佳RMS
    best_idx = df['rms_num'].idxmin()
    best_epoch = df.loc[best_idx, 'epoch_num']
    best_rms = df.loc[best_idx, 'rms_num']
    
    def init():
        line_rms.set_data([], [])
        line_gamma.set_data([], [])
        line_delta.set_data([], [])
        line_diff.set_data([], [])
        return [line_rms, line_gamma, line_delta, line_diff]
    
    def update(frame):
        i = indices[frame]
        line_rms.set_data(epochs[:i+1], rms_values[:i+1])
        line_gamma.set_data(epochs[:i+1], gamma_values[:i+1])
        line_delta.set_data(epochs[:i+1], delta_values[:i+1])
        line_diff.set_data(epochs[:i+1], df['diff_num'].values[:i+1])
        
        # 更新最佳点标记
        if i >= best_idx:
            best_marker.set_offsets([[best_epoch, best_rms]])
        
        fig.suptitle(f'Epoch {epochs[i]} | RMS: {rms_values[i]:.4f}', fontsize=12)
        return [line_rms, line_gamma, line_delta, line_diff]
    
    ani = animation.FuncAnimation(fig, update, frames=len(indices), 
                                  init_func=init, blit=True, interval=1000//fps)
    
    if save_path:
        # 保存为GIF
        frames = []
        for idx in indices:
            update(indices.index(idx))
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=80, bbox_inches='tight')
            buf.seek(0)
            frames.append(Image.open(buf).copy())
            buf.close()
        
        # 转换为RGB模式并保存
        frames[0].save(save_path, save_all=True, append_images=frames[1:], 
                      duration=1000//fps, loop=0)
        print(f"✓ 动画已保存: {save_path}")
    
    plt.close(fig)
    return ani

def create_zernike_animation(df, save_path=None, fps=10, skip_frames=1):
    """创建Zernike系数演化动画"""
    coef_matrix = extract_zernike_matrix(df)
    if coef_matrix is None:
        print("无法解析Zernike系数")
        return None
    
    epochs = df['epoch_num'].values
    n_modes = coef_matrix.shape[1]
    
    # 降采样
    indices = list(range(0, len(epochs), skip_frames))
    if indices[-1] != len(epochs) - 1:
        indices.append(len(epochs) - 1)
    
    fig, axes = plt.subplots(3, 5, figsize=(15, 9))
    axes = axes.flatten()
    
    lines = []
    for i in range(min(n_modes, 15)):
        ax = axes[i]
        line, = ax.plot([], [], linewidth=0.8)
        lines.append(line)
        ax.set_xlim(0, max(epochs))
        vmax = max(abs(coef_matrix[:, i].min()), abs(coef_matrix[:, i].max()))
        if vmax > 0:
            ax.set_ylim(-vmax*1.2, vmax*1.2)
        ax.set_title(f'Z{i+1}')
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
    
    # 隐藏多余子图
    for i in range(n_modes, len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    
    def init():
        for line in lines:
            line.set_data([], [])
        return lines
    
    def update(frame):
        i = indices[frame]
        for j in range(min(n_modes, 15)):
            lines[j].set_data(epochs[:i+1], coef_matrix[:i+1, j])
        fig.suptitle(f'Epoch {epochs[i]}', fontsize=14)
        return lines
    
    ani = animation.FuncAnimation(fig, update, frames=len(indices),
                                  init_func=init, blit=True, interval=1000//fps)
    
    if save_path:
        frames = []
        for idx in indices:
            update(indices.index(idx))
            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=80, bbox_inches='tight')
            buf.seek(0)
            frames.append(Image.open(buf).copy())
            buf.close()
        
        frames[0].save(save_path, save_all=True, append_images=frames[1:],
                      duration=1000//fps, loop=0)
        print(f"✓ 动画已保存: {save_path}")
    
    plt.close(fig)
    return ani

In [ ]:
# 动画控制面板
def show_animation_panel():
    """显示动画控制面板"""
    print("=" * 60)
    print("动画与GIF导出")
    print("=" * 60)
    print("使用方法:")
    print("  create_rms_animation(df, save_path='rms.gif', fps=10)")
    print("  create_zernike_animation(df, save_path='zernike.gif', fps=10)")
    print("\n参数说明:")
    print("  - df: 加载的DataFrame数据")
    print("  - save_path: GIF保存路径 (可选，默认不保存)")
    print("  - fps: 帧率 (默认10)")
    print("  - skip_frames: 跳帧数 (用于减少总帧数，默认1)")
    print("=" * 60)
    
    # 显示动画预览
    if current_df['df'] is not None:
        print("\n正在生成动画预览...")
        # 在notebook中显示动画
        from IPython.display import HTML
        
        # RMS动画
        fig, axes = plt.subplots(2, 2, figsize=(10, 7))
        epochs = current_df['df']['epoch_num'].values
        rms_values = current_df['df']['rms_num'].values
        
        # 动态更新示例 - 只显示关键帧
        n_frames = min(30, len(epochs))
        step = len(epochs) // n_frames
        key_epochs = epochs[::step][:n_frames]
        key_rms = rms_values[::step][:n_frames]
        
        for i, (ep, rm) in enumerate(zip(key_epochs, key_rms)):
            axes[0, 0].scatter(ep, rm, c='blue', s=30, alpha=0.7)
        axes[0, 0].plot(epochs, rms_values, 'b-', alpha=0.3, linewidth=0.5)
        axes[0, 0].set_xlabel('Epoch')
  

In [ ]:
# 快速生成GIF的按钮
def quick_gif_animation(fps=15, skip=5, save_rms=True, save_zernike=True):
    """快速生成GIF动画"""
    if current_df['df'] is None:
        print("请先加载数据")
        return
    
    df = current_df['df']
    run_name = current_df.get('run_name', 'run')[:8]
    output_dir = Path('../data/rms_zernike')
    
    if save_rms:
        save_path = output_dir / f'{run_name}_rms_anim.gif'
        print(f"生成RMS动画: {save_path}")
        create_rms_animation(df, save_path=str(save_path), fps=fps, skip_frames=skip)
    
    if save_zernike:
        save_path = output_dir / f'{run_name}_zernike_anim.gif'
        print(f"生成Zernike动画: {save_path}")
        create_zernike_animation(df, save_path=str(save_path), fps=fps, skip_frames=skip)
    
    print("\n✓ 全部完成!")

gif_btn = widgets.Button(description='生成GIF动画', button_style='danger')
output_gif = widgets.Output()

def on_gif_click(b):
    with output_gif:
        clear_output()
        if current_df['df'] is not None:
            quick_gif_animation(fps=15, skip=5)
        else:
            print("请先加载数据")

gif_btn.on_click(on_gif_click)
display(gif_btn)
display(output_gif)

In [ ]:
# 播放内联动画预览
def play_inline_animation(df, fps=10):
    """在notebook中内联播放动画预览"""
    from IPython.display import HTML
    
    # 使用matplotlib动画
    ani = create_rms_animation(df, save_path=None, fps=fps, skip_frames=max(1, len(df)//50))
    
    # 转换为HTML5视频
    plt.close('all')
    return ani

play_btn = widgets.Button(description='播放动画预览', button_style='success')
output_play = widgets.Output()

def on_play_click(b):
    with output_play:
        clear_output()
        if current_df['df'] is not None:
            print("正在生成动画 (可能需要几秒)...")
            # 显示静态预览
            fig, ax = plt.subplots(1, 1, figsize=(10, 5))
            epochs = current_df['df']['epoch_num'].values
            rms_values = current_df['df']['rms_num'].values
            ax.plot(epochs, rms_values, 'b-', linewidth=1)
            ax.fill_between(epochs, rms_values, alpha=0.3)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('RMS')
            ax.set_title(f'SPGD优化 - RMS历史 (共{len(df)} epochs)')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            print("提示: 点击上方「生成GIF动画」按钮保存为GIF文件")
        else:
            print("请先加载数据")

play_btn.on_click(on_play_click)
display(play_btn)
display(output_play)